🚀 Excellent. We'll first improve the API properly, then move into optimization-focused Day 12.

✅ Mini Improvement — Dynamic Top-N Recommendations

Right now:

Always returns top 10

But real APIs allow:

User-controlled limits

Example:

/recommend/1?n=5

or

/recommend/1?n=20


---

Step 1 — Update endpoint

📁 api/app.py

Replace:

@app.get("/recommend/{user_id}")

def recommend_movies(
    user_id:int
):

with:

@app.get("/recommend/{user_id}")

def recommend_movies(
    user_id:int,
    n:int=10
):


---

Step 2 — Slice recommendations dynamically

Replace:

recommendations = hybrid_recommendation(

with:

recommendations = hybrid_recommendation(
    user_id,
    liked_movies,
    ratings,
    movies,
    similarity,
    model
).head(n)


---

Step 3 — Test

Examples:

http://127.0.0.1:8000/recommend/1?n=5

http://127.0.0.1:8000/recommend/1?n=3

http://127.0.0.1:8000/recommend/1?n=15


---

🧠 Why this matters

You just added:

Parameterized API behavior

instead of hardcoded logic.

That’s how production APIs are usually designed.


---

# 🚀 Day 12 — API Optimization + Better Response Design

Now your API works.

Next question:

Can we make it faster
and cleaner?

Today we improve:

✅ API efficiency
✅ Response structure
✅ Input validation
✅ Better debugging
✅ Better API design


---

🚀 Problem 1 — Slow recommendation generation

Right now:

Every request does:

Loop through ALL movies
↓
Predict scores one-by-one
↓
Build DataFrames
↓
Sort everything

Works for MovieLens 100K.

But for:

Netflix-scale datasets

this becomes expensive.


---

First optimization mindset

Currently:

for movie in all_movies:

Production systems often use:

Candidate generation
↓
Ranking

instead of scoring entire catalog.

You are beginning to enter recommendation system scaling concepts.


---

🚀 Improvement 1 — Add timing logs

Let's measure response time.


---

Step 1 — Import time

📁 api/app.py

import time


---

Step 2 — Measure execution

Inside endpoint:

start = time.time()

Before return:

end = time.time()

print(
    f"Response time: {end-start:.2f} sec"
)


---

Full example

@app.get("/recommend/{user_id}")

def recommend_movies(
    user_id:int,
    n:int=10
):

    start = time.time()

    liked_movies = df[
        (
            df['user_id']==user_id
        )
    ].sort_values(
        'rating',
        ascending=False
    )['title'].head(5).tolist()

    recommendations = hybrid_recommendation(
        user_id,
        liked_movies,
        ratings,
        movies,
        similarity,
        model
    ).head(n)

    output=[]

    for _,row in recommendations.iterrows():

        movie_name = movies[
            movies['movie_id']==row['movie_id']
        ]['title'].values[0]

        output.append({

            "movie":movie_name,

            "score":float(
                row['final_score']
            )
        })

    end = time.time()

    print(
        f"Response time: {end-start:.2f} sec"
    )

    return output


---

🧠 Why timing matters

Production ML systems care about:

Latency

Example:

100 ms
vs
5 seconds

Huge difference for user experience.


---

🚀 Improvement 2 — Better response structure

Currently response:

[
  {
    "movie":"Titanic",
    "score":0.91
  }
]

Better API design:

{
  "user_id":1,
  "recommendation_count":5,
  "recommendations":[
    {
      "movie":"Titanic",
      "score":0.91
    }
  ]
}


---

Update return

Replace:

return output

with:

return {

    "user_id":user_id,

    "recommendation_count":len(output),

    "recommendations":output
}


---

🚀 Improvement 3 — Validate user IDs

Problem:

/recommend/999999

may crash or behave strangely.


---

Add validation

After:

liked_movies = ...

Add:

if len(liked_movies)==0:

    return {

        "error":
        "User not found"
    }


---

🚀 Improvement 4 — Add health endpoint

Very common in production systems.


---

Add endpoint

@app.get("/health")

def health():

    return {

        "status":"healthy"
    }


---

Why health endpoints matter

Docker/Kubernetes/cloud systems use them to check:

Is API alive?


---

🚀 Improvement 5 — Add recommendation metadata

Add:

"model_version":"svd_model_v1"

Example response:

{
  "user_id":1,
  "model_version":"svd_model_v1",
  "recommendations":[]
}

This becomes useful later for:

A/B testing
rollback
monitoring


---

🧠 Important production concepts learned today

1. API parameterization

?n=5


---

2. Latency measurement

Response timing


---

3. Response schema design

Structured JSON responses.


---

4. Input validation

Prevent invalid requests.


---

5. Health monitoring

Used heavily in deployments.


---

Current architecture maturity

You now have:

✅ ML recommendation engine
✅ Hybrid recommender
✅ Versioned artifacts
✅ FastAPI backend
✅ Dynamic API parameters
✅ Timing metrics
✅ Validation
✅ Health monitoring

This is becoming very close to a real deployable ML service.


---

🎯 Homework

1. Test invalid users

Example:

/recommend/999999

Check:

Does API return clean error?


---

2. Compare response time

Try:

n=5
n=20
n=50

Observe timing.


---

3. Open Swagger docs again

/docs

Notice:

FastAPI automatically documents parameters

including:

user_id
n



Excellent debugging again 👀
Both are real production/backend issues.

Let's solve them one-by-one.


---

🚀 Issue 1 — Invalid user causing JSON serialization error

Main error:

ValueError:
Out of range float values are not JSON compliant

This means:

NaN / infinity

is entering your JSON response.


---

Why this happens

For invalid users:

liked_movies = []

Then inside:

hybrid_recommendation()

some calculations produce:

NaN

Especially during:

MinMaxScaler()

or score normalization.

Then:

float(np.nan)

cannot be converted to valid JSON.


---

🚀 BEST FIX (Better than exception handling)

Instead of waiting for failure:

Validate BEFORE recommendation generation.

This is actually better API design.


---

✅ Proper validation

Add this BEFORE:

recommendations = hybrid_recommendation(...)


---

Replace this

liked_movies = df[

with:

user_data = df[
    df['user_id'] == user_id
]

if user_data.empty:

    return {

        "status":"error",

        "message":"Invalid user_id"
    }

liked_movies = user_data.sort_values(
    'rating',
    ascending=False
)['title'].head(5).tolist()


---

Why this is better than try-except

Because:

Invalid user

is NOT really an exception.

It's:

Bad client input

Production APIs usually:

✅ validate inputs first
✅ reserve exceptions for unexpected failures


---

🚀 Better exception block too

Currently:

return {'error':str(e)}

Better:

return {

    "status":"failed",

    "error":str(e)
}


---

🚀 Issue 2 — Models loading twice in uvicorn

This is VERY normal.

And excellent observation 👀


---

Why it happens

You are using:

uvicorn api.app:app --reload

The --reload flag creates:

Process 1

File watcher

and

Process 2

Actual server

Both import your module.

So:

model = load_latest_svd_model()

runs twice.


---

That’s why you see:

Loaded: model/svd_model_v1.pkl
Loaded: model/similarity_v1.pkl

two times.


---

🚀 Production-style fix

Instead of loading globally:

model = load_latest_svd_model()

load during:

startup event


---

✅ Better FastAPI architecture


---

Step 1 — Create app state

Replace:

model, model_name = load_latest_svd_model()

similarity = load_latest_similarity()

REMOVE them.


---

Step 2 — Add startup event

@app.on_event("startup")

def load_resources():

    app.state.model, app.state.model_name = load_latest_svd_model()

    app.state.similarity = load_latest_similarity()

    print("Resources loaded...")


---

Step 3 — Access resources from app.state

Inside recommend endpoint:

Replace:

model

with:

app.state.model

Replace:

similarity

with:

app.state.similarity

Replace:

model_name

with:

app.state.model_name


---

Example

recommendations = hybrid_recommendation(

    user_id,

    liked_movies,

    ratings,

    movies,

    app.state.similarity,

    app.state.model

).head(n)


---

Health endpoint also

Replace:

model is not None

with:

app.state.model is not None


---

🧠 Why startup events matter

Production APIs usually:

Load heavy resources once
during startup

instead of during import.

This becomes VERY important later for:

✅ Docker
✅ Kubernetes
✅ cloud deployment
✅ memory optimization


---

🧠 What you just discovered naturally

1. Input validation vs exception handling


2. JSON serialization edge cases


3. NaN propagation in ML pipelines


4. Uvicorn reload architecture


5. FastAPI lifecycle management


6. Application state management



These are genuinely real backend engineering concepts.



Exactly — when you run:

```bash
uvicorn app:app --reload
```

Uvicorn actually spins up **two processes** under the hood:

---

### 🔹 1. File Watcher Process
- This is the **“reload supervisor”**.
- It watches your project files for changes (using libraries like `watchgod` or `watchdog`).
- If you edit a `.py` file, the watcher detects it and restarts the server automatically.
- This process doesn’t serve requests — it just monitors your code.

---

### 🔹 2. Actual Server Process
- This is the **worker process** that runs your FastAPI app.
- It’s the one listening on `http://127.0.0.1:8000` (or whatever port you set).
- When the watcher sees a file change, it kills this process and spawns a new one with the updated code.

---

### ✅ Why two processes?
- Separation of concerns:
  - One process is dedicated to **watching files**.
  - The other is dedicated to **serving requests**.
- This makes hot‑reload possible without mixing file‑watching logic into the server itself.
- It also ensures that if your app crashes, the watcher can restart it.

---

### 🔹 Without `--reload`
- Only the server process runs.
- No file watcher, so you must manually restart the server after code changes.

---

👉 So in short:  
- **File watcher** = supervisor, monitors code changes.  
- **Server process** = runs your FastAPI app, handles requests.  

That’s why you see two processes in the background when using `--reload`.  

Would you like me to also show you how to **disable reload but still run multiple workers** (using `--workers`), so you can see the difference between reload vs scaling?

Exactly — let’s break down what happens when you run Uvicorn with different options:

---

### 🔹 `uvicorn app:app --reload`
- **Two processes**:
  1. **File watcher (supervisor)** → monitors your Python files for changes.
  2. **Server process** → runs your FastAPI app.
- If you edit a file, the watcher kills and restarts the server process.
- This is great for development, but not recommended for production because of the overhead.

---

### 🔹 `uvicorn app:app --workers 4`
- **Multiple worker processes** (here, 4).
- Each worker runs your FastAPI app independently, all listening on the same port.
- The OS load‑balances incoming requests across workers.
- This is for **scaling** — handling more concurrent requests.
- No file watcher here, so you must restart manually if you change code.

---

### 🔹 `uvicorn app:app --reload --workers 4`
- Not allowed — `--reload` and `--workers` conflict.
- Reload mode always runs with a single worker, because the watcher can only restart one process cleanly.

---

### ✅ Summary
- `--reload` → development convenience (auto‑restart on code changes).
- `--workers` → production scaling (multiple processes to handle traffic).
- They serve different purposes:
  - Reload = **developer productivity**.
  - Workers = **performance under load**.

---

👉 Think of it like this:
- **Reload** = “I’m coding, restart when I save.”  
- **Workers** = “I’m deploying, handle more users at once.”  

Do you want me to show you a **recommended production setup** (using Uvicorn + Gunicorn with workers) so you can see how people usually run FastAPI apps in real deployments?

Perfect — let’s walk through how a **production‑ready FastAPI deployment** is usually set up.  

---

### 🔹 Why not just `uvicorn app:app --reload`?
- `--reload` is **only for development**.  
- It uses a file watcher + single worker process.  
- Not efficient or stable under heavy traffic.  

---

### 🔹 Production Setup (Uvicorn + Gunicorn)
The common pattern is to run **Gunicorn** as a process manager, and let it spawn multiple **Uvicorn workers**:

```bash
gunicorn -w 4 -k uvicorn.workers.UvicornWorker app:app
```

- `-w 4` → number of worker processes (tune based on CPU cores).  
- `-k uvicorn.workers.UvicornWorker` → tells Gunicorn to use Uvicorn workers.  
- `app:app` → your FastAPI app object.  

---

### 🔹 Why Gunicorn?
- Handles **process management** (spawning, monitoring, restarting workers).  
- Provides **graceful restarts** and better logging.  
- Supports **scaling** across multiple CPU cores.  
- Uvicorn alone can run multiple workers (`--workers`), but Gunicorn is more robust for production.

---

### 🔹 Typical Deployment Flow
1. **Local development**  
   ```bash
   uvicorn app:app --reload
   ```
   → auto‑reload on code changes.

2. **Production (single server)**  
   ```bash
   gunicorn -w 4 -k uvicorn.workers.UvicornWorker app:app
   ```
   → multiple workers, no reload.

3. **Production (with reverse proxy)**  
   - Put **Nginx** or **Traefik** in front of Gunicorn/Uvicorn.  
   - Proxy handles SSL, load balancing, static files.  
   - Gunicorn/Uvicorn only serve API requests.

---

### 🔹 Example with Nginx
- Nginx listens on port 80/443.  
- Forwards requests to Gunicorn running on port 8000.  
- Gunicorn spawns multiple Uvicorn workers.  

---

✅ **Summary:**  
- `--reload` → dev mode, auto restart.  
- `--workers` → production scaling.  
- Gunicorn + Uvicorn workers → recommended production setup.  
- Reverse proxy (Nginx/Traefik) → handles SSL, routing, static files.

---

Do you want me to sketch out a **sample Nginx config** that shows how it proxies requests to your FastAPI app? That’s usually the next step when moving from dev to production.

Here’s a simple **Nginx reverse proxy config** that shows how you’d put Nginx in front of your FastAPI app (running with Gunicorn + Uvicorn workers):

```nginx
server {
    listen 80;
    server_name yourdomain.com;

    # Redirect all HTTP to HTTPS
    return 301 https://$host$request_uri;
}

server {
    listen 443 ssl;
    server_name yourdomain.com;

    # SSL certificates (from Let's Encrypt or similar)
    ssl_certificate /etc/letsencrypt/live/yourdomain.com/fullchain.pem;
    ssl_certificate_key /etc/letsencrypt/live/yourdomain.com/privkey.pem;

    # Proxy settings
    location / {
        proxy_pass http://127.0.0.1:8000;   # Gunicorn/Uvicorn backend
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;
    }

    # Optional: serve static files directly
    location /static/ {
        alias /path/to/your/app/static/;
    }
}
```

---

### 🔹 How this works
- **Nginx** listens on port 80/443 (public web).  
- It forwards requests to **Gunicorn** running on `127.0.0.1:8000`.  
- Gunicorn spawns multiple **Uvicorn workers** to handle traffic.  
- Nginx handles SSL, compression, caching, and static files.  
- Your FastAPI app only focuses on API logic.

---

### 🔹 Typical Run Command
Start Gunicorn with Uvicorn workers:
```bash
gunicorn -w 4 -k uvicorn.workers.UvicornWorker app:app --bind 127.0.0.1:8000
```

---

### ✅ Summary
- **Development:** `uvicorn app:app --reload`  
- **Production (simple):** `uvicorn app:app --workers 4`  
- **Production (robust):** `gunicorn -w 4 -k uvicorn.workers.UvicornWorker app:app` behind **Nginx**.  
- Nginx handles SSL, static files, and reverse proxying. Gunicorn manages workers. Uvicorn runs the actual FastAPI app.

---

Would you like me to also sketch out a **systemd service file** for Gunicorn, so it runs automatically on server boot and restarts if it crashes? That’s the next piece in a real deployment.